In [1]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("data/raw")
PROCESSED_PATH = Path("data/processed")


def parse_date_columns(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    """Convert any column whose name contains 'date' to a datetime dtype."""
    for col in df.columns:
        if "date" in col.lower():
            try:
                df[col] = pd.to_datetime(df[col])
            except (ValueError, TypeError) as exc:
                print(f"  [warn] Could not parse '{col}' in {filename} as a date: {exc}")
    return df


def handle_missing_values(df: pd.DataFrame, filename: str) -> pd.DataFrame:
    """Fill missing values and report what was changed."""
    missing_before = df.isnull().sum()
    missing_cols = missing_before[missing_before > 0]

    if missing_cols.empty:
        print(f"  No missing values found in {filename}")
        return df

    print(f"  Missing values found in {filename}:")
    for col, count in missing_cols.items():
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].median()
            df[col] = df[col].fillna(fill_value)
            print(f"    - {col}: {count} missing -> filled with median ({fill_value})")
        else:
            df[col] = df[col].fillna("Unknown")
            print(f"    - {col}: {count} missing -> filled with 'Unknown'")

    return df


def clean_file(file: Path) -> pd.DataFrame:
    df = pd.read_csv(file)

    before_rows = len(df)
    df = df.drop_duplicates()
    duplicates_removed = before_rows - len(df)
    if duplicates_removed:
        print(f"  Removed {duplicates_removed} duplicate row(s)")

    df = parse_date_columns(df, file.name)
    df = handle_missing_values(df, file.name)

    return df


def main():
    PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

    csv_files = sorted(RAW_PATH.glob("*.csv"))
    print(f"Found {len(csv_files)} raw CSV file(s) in {RAW_PATH}\n")

    for file in csv_files:
        print(f"Processing {file.name}")
        df = clean_file(file)

        cleaned_name = file.stem + "_cleaned.csv"
        df.to_csv(PROCESSED_PATH / cleaned_name, index=False)
        print(f"  Saved -> {PROCESSED_PATH / cleaned_name}\n")

    print("All files cleaned successfully!")


if __name__ == "__main__":
    main()


Found 0 raw CSV file(s) in data\raw

All files cleaned successfully!
